# CRISP-DM Phase 4: Modeling (Medical Dataset)

We load **processed** matrices from Phase 3 (`X_train`, `X_test`, `y_train`, `y_test`) plus `metadata.json`.

**Goal:** Predict `Result` with **positive class = 1** (`positive`).

**Clinical priority:** Maximize **Recall (sensitivity)** for the positive class — false negatives are typically more costly than false positives in screening-style framing.

**Protocol (reduces methodological error):**
1. **Stratified K-fold CV** on the training set only → report all candidates; **mean CV Recall** identifies `cv_winner`.
2. **Deploy policy:** export **Gradient Boosting** by default (override variable in the code cell if you want CV-winner-only).
3. Refit all candidates on the **full** training set for comparison tables.
4. **Single** hold-out report on `X_test` (not used for picking the winner).
5. **Learning curve** (train only): bias–variance vs training size.
6. **Calibration + Brier** on hold-out: probability reliability check.
7. Export the deploy model (+ JSON sidecar with scores and CV/deploy lineage).
8. **MLP (feed-forward NN):** `sklearn.neural_network.MLPClassifier` with **early stopping**, same CV/hold-out protocol as baselines; **no duplicate data** beyond Phase 3 CSVs.



In [ ]:
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, learning_curve
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')


## 1. Load processed data and metadata

Paths resolve whether you run the notebook from the project root or from `classification_Medical_data _set/`.
Column order is enforced from `metadata.json` when available.

**Data hygiene:** Reads **only** `data/processed/medical/*.csv` (Phase 3). `Medicaldataset.csv` is used **only** to find `base_dir`, not re-loaded into modeling.


In [ ]:
candidate_roots = [Path.cwd(), Path.cwd() / 'classification_Medical_data _set']
base_dir = next((r for r in candidate_roots if (r / 'Medicaldataset.csv').exists()), None)
if base_dir is None:
    raise FileNotFoundError("Could not locate Medicaldataset.csv; run from project root or classification_Medical_data _set.")

data_dir = base_dir / 'data' / 'processed' / 'medical'
meta_path = data_dir / 'metadata.json'

with open(meta_path, encoding='utf-8') as f:
    metadata = json.load(f)

X_train = pd.read_csv(data_dir / 'X_train.csv')
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv').squeeze('columns')
y_test = pd.read_csv(data_dir / 'y_test.csv').squeeze('columns')

if 'continuous_columns' in metadata and 'flag_columns' in metadata:
    feature_order = metadata['continuous_columns'] + metadata['flag_columns']
    missing = [c for c in feature_order if c not in X_train.columns]
    if missing:
        raise ValueError(f'Processed X_train is missing expected columns: {missing}')
    X_train = X_train[feature_order]
    X_test = X_test[feature_order]

POS_LABEL = 1
if not set(np.unique(y_train)).issubset({0, 1}) or not set(np.unique(y_test)).issubset({0, 1}):
    raise ValueError('y must be binary 0/1 (negative/positive).')

print(f'Canonical processed tensors (Phase 3 only): {data_dir.resolve()}')
print(f'Base directory: {base_dir}')
print(f'Training: {X_train.shape}, Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.4f}, Test positive rate: {y_test.mean():.4f}')
print(f'Target mapping (from metadata): {metadata.get("target_mapping")}')


### Five-supervisor gate — adding `MLPClassifier`

| Voice | Check | Pass |
|--------|--------|------|
| **Domain / ethics** | NN is one candidate among interpretable baselines. | Deploy policy stays **unless** you change `DEPLOY_MODEL_NAME`. |
| **Data / leakage** | Same stratified split as everyone else. | **Single** matrix source: processed CSVs only. |
| **Method** | Features scaled in `01b`; shallow MLP + early stopping for small *n*. | Architecture fixed in code for audit. |
| **Class balance** | MLP has **no** built-in `class_weight`. | Compare **recall** honestly vs GBM / logistic. |
| **Repro** | `random_state` + plots in §4b. | Loss + ROC on hold-out documented below.

## 2. Model zoo

- **Dummy (stratified)** — sanity baseline (preserves label prevalence).
- **Logistic Regression** — linear, interpretable baseline (`class_weight='balanced'` optional emphasis on minority).
- **Random Forest** — ensemble trees; strong default on tabular data.
- **SVC (RBF)** — kernel classifier (requires scaled features — satisfied by Phase 3).
- **Gradient Boosting** — sequential trees; often strong on tabular data.

**Positive class for all metrics:** `1` = `positive`.


In [ ]:
RANDOM_STATE = 42
CV_SPLITS = 5

model_candidates = {
    'Dummy (stratified)': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced', solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', min_samples_leaf=2
    ),
    'SVC (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, class_weight='balanced'),
    'MLP (neural net)': MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, random_state=RANDOM_STATE, max_depth=3, learning_rate=0.1
    ),
}


## 3. Stratified cross-validation (train only) + hold-out test

**Model comparison:** report mean **CV Recall** on `y_train == 1` for every candidate (excluding the dummy from the *winner* role).

**Deploy model:** by default we export **Gradient Boosting** when it is in the zoo (`DEPLOY_MODEL_NAME`); the true CV recall leader is stored as `cv_winner` and echoed in `medical_model_selection.json`.

**Report:** one pass of test metrics after refitting on the full training set (for tables/plots — not for cherry-picking the deploy choice).

In [ ]:
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {'recall': 'recall', 'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}

cv_rows = []
for name, est in model_candidates.items():
    out = cross_validate(
        clone(est),
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    cv_rows.append({
        'Model': name,
        'CV Recall mean': out['test_recall'].mean(),
        'CV Recall std': out['test_recall'].std(),
        'CV PR-AUC mean': out['test_average_precision'].mean(),
        'CV PR-AUC std': out['test_average_precision'].std(),
        'CV ROC-AUC mean': out['test_roc_auc'].mean(),
        'CV ROC-AUC std': out['test_roc_auc'].std(),
    })

df_cv = pd.DataFrame(cv_rows).set_index('Model').sort_values('CV Recall mean', ascending=False)
print('Stratified cross-validation on training set only')
display(df_cv.style.highlight_max(axis=0, subset=['CV Recall mean', 'CV PR-AUC mean', 'CV ROC-AUC mean']))

non_dummy = [n for n in df_cv.index if not n.lower().startswith('dummy')]
cv_winner = df_cv.loc[non_dummy, 'CV Recall mean'].idxmax()
# Project policy: ship Gradient Boosting for deployment (set to None to export CV winner only)
DEPLOY_MODEL_NAME = 'Gradient Boosting'
best_name = DEPLOY_MODEL_NAME if DEPLOY_MODEL_NAME in model_candidates else cv_winner
if best_name != cv_winner:
    print(f'CV winner by mean recall: **{cv_winner}**')
    print(f'Deploy policy: export **{best_name}** (override).\n')
else:
    print(f'\nSelected for export: **{best_name}** (matches CV winner).\n')

# Refit all on full training data; evaluate once on hold-out test
fitted_models = {}
predictions = {}
test_rows = []

for name, est in model_candidates.items():
    m = clone(est)
    m.fit(X_train, y_train)
    fitted_models[name] = m
    y_pred = m.predict(X_test)
    predictions[name] = y_pred
    y_prob = m.predict_proba(X_test)[:, 1]

    test_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (pos)': precision_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        'Recall (pos)': recall_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        'F1 (pos)': f1_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'PR-AUC': average_precision_score(y_test, y_prob),
    })

df_test = pd.DataFrame(test_rows).set_index('Model').sort_values('Recall (pos)', ascending=False)
print('Hold-out test metrics (single evaluation; primary model fixed above)')
display(df_test.style.highlight_max(axis=0, subset=['Recall (pos)', 'PR-AUC', 'ROC-AUC']))

best_model = fitted_models[best_name]
print(f'Deploy model refit on full train: {best_name}')
print(f'Hold-out recall (pos): {df_test.loc[best_name, "Recall (pos)"]:.4f}')

## 4. Learning curve (train-only diagnostic)

**Purpose:** See how **train vs cross-validated PR-AUC** changes as we use more training rows. A large gap (train high, CV much lower) suggests **high variance** (overfitting to training noise at this sample size). Parallel trajectories that plateau suggest **bias** or **sufficient data**.

This complements our fixed-size CV table: it answers *“would more data likely help?”* and *“is there a train–validation gap?”* across training sizes.

**Scope:** Fitted only on `X_train` / `y_train` — the hold-out test is **not** used here.

In [ ]:
lc_est = clone(model_candidates[best_name])
lc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_sizes_abs, train_scores, val_scores = learning_curve(
    lc_est,
    X_train,
    y_train,
    train_sizes=np.linspace(0.2, 1.0, 6, dtype=float),
    cv=lc_cv,
    scoring='average_precision',
    n_jobs=-1,
)
train_m = train_scores.mean(axis=1)
train_s = train_scores.std(axis=1)
val_m = val_scores.mean(axis=1)
val_s = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes_abs, train_m, 'o-', label='Train PR-AUC', color='C0')
ax.fill_between(train_sizes_abs, train_m - train_s, train_m + train_s, alpha=0.15, color='C0')
ax.plot(train_sizes_abs, val_m, 'o-', label='CV PR-AUC', color='C1')
ax.fill_between(train_sizes_abs, val_m - val_s, val_m + val_s, alpha=0.15, color='C1')
ax.set_xlabel('Training samples')
ax.set_ylabel('Average precision (PR-AUC)')
ax.set_title(f'Learning curve — {best_name}')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4b. Neural net diagnostics (MLP)

**Left:** MLP **loss** across iterations (early stopping when enabled).  
**Right:** **ROC** on the **hold-out** for every non-dummy model with probabilities.


In [ ]:
from sklearn.metrics import auc, roc_curve

mlp_model = fitted_models.get("MLP (neural net)")
fig, (ax_loss, ax_roc) = plt.subplots(1, 2, figsize=(13, 5))

if mlp_model is not None and getattr(mlp_model, "loss_curve_", None) is not None:
    ax_loss.plot(np.arange(len(mlp_model.loss_curve_)), mlp_model.loss_curve_, color="C2", lw=2)
    ax_loss.set_xlabel("Iteration")
    ax_loss.set_ylabel("Loss")
    ax_loss.set_title("MLP loss curve (train split)")
    ax_loss.grid(True, alpha=0.3)
else:
    ax_loss.text(0.5, 0.5, "No loss_curve_ available.", ha="center", va="center")
    ax_loss.set_axis_off()

for name, model in fitted_models.items():
    if "Dummy" in name:
        continue
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax_roc.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc(fpr, tpr):.3f})")

ax_roc.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate")
ax_roc.set_title("ROC — hold-out test")
ax_roc.legend(loc="lower right", fontsize=8)
ax_roc.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()


## 5. Calibration on hold-out test (reliability check)

**Discrimination** (ROC-AUC, PR-AUC) asks how well scores **rank** cases. **Calibration** asks whether predicted probabilities match **observed** event rates in bins (roughly: “when the model says 30%, is it ~30%?”).

We plot a **reliability diagram** and report **Brier score** on `X_test` for the **deploy model only**. This does **not** replace external validation; it is a standard transparency step for binary classifiers.

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]
brier = brier_score_loss(y_test, y_proba)
print(f'Brier score (lower is better): {brier:.4f}')

prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='uniform')
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred, prob_true, 's-', label=best_name)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual)')
ax.set_title('Calibration curve (hold-out test)')
ax.legend(loc='best')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()

## 6. Confusion matrices (hold-out test)

Each matrix uses predictions from models **refit on the full training set**.  
Rows: actual (`0` = negative, `1` = positive); columns: predicted. 
- **Top Left**: True Negatives (Healthy predicted healthy)
- **Bottom Right**: True Positives (Sick predicted sick)
- **Bottom Left**: False Negatives (Sick predicted healthy - DANGEROUS!)
- **Top Right**: False Positives (Healthy predicted sick - Annoying but safe)

In [ ]:
names = list(predictions.keys())
n = len(names)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for i, name in enumerate(names):
    cm = confusion_matrix(y_test, predictions[name], labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[i],
        cbar=False,
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['Actual 0', 'Actual 1'],
    )
    axes[i].set_title(f'{name}')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## 7. Feature importance (Random Forest)

Tree ensembles expose **impurity-based importance** (higher = more reduction in Gini impurity across splits).  
Use as an explanatory aid, not a causal clinical claim.

In [ ]:
rf_fitted = fitted_models['Random Forest']
importances = rf_fitted.feature_importances_
indices = np.argsort(importances)[::-1]
names_ord = [X_train.columns[i] for i in indices]

plt.figure(figsize=(10, 6))
plt.barh(names_ord[::-1], importances[indices][::-1], color='steelblue')
plt.title('Random Forest feature importance (refit on full training set)')
plt.xlabel('Relative importance (Gini-based)')
plt.tight_layout()
plt.show()

## 8. Export the deploy model

We serialize **`best_name`**, which by default follows a **project deploy policy** (`Gradient Boosting` when present in the zoo). CV still reports all candidates; `cv_winner_by_mean_recall` in the JSON records who “won” recall on the training folds.

`medical_rf_model.pkl` is also written **only if** the selected model is Random Forest — otherwise the canonical artifact is `medical_best_model.pkl`.

Downstream code should load `medical_model_selection.json` to know which file is authoritative for this run.

In [ ]:
models_dir = base_dir / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

best_path = models_dir / 'medical_best_model.pkl'
joblib.dump(best_model, best_path)

test_brier = float(brier_score_loss(y_test, best_model.predict_proba(X_test)[:, 1]))
selection = {
    'selected_model': best_name,
    'cv_winner_by_mean_recall': cv_winner,
    'deploy_model_choice': DEPLOY_MODEL_NAME,
    'cv_recall_mean': float(df_cv.loc[best_name, 'CV Recall mean']),
    'cv_recall_std': float(df_cv.loc[best_name, 'CV Recall std']),
    'test_recall_pos': float(df_test.loc[best_name, 'Recall (pos)']),
    'test_pr_auc': float(df_test.loc[best_name, 'PR-AUC']),
    'test_roc_auc': float(df_test.loc[best_name, 'ROC-AUC']),
    'test_brier': test_brier,
    'artifact': str(best_path.resolve()),
    'random_state': RANDOM_STATE,
    'cv_splits': CV_SPLITS,
}
with open(models_dir / 'medical_model_selection.json', 'w', encoding='utf-8') as f:
    json.dump(selection, f, indent=2)

if best_name == 'Random Forest':
    rf_alias = models_dir / 'medical_rf_model.pkl'
    joblib.dump(best_model, rf_alias)
    print(f'Also saved alias for backward compatibility: {rf_alias.resolve()}')

print(f'Saved deploy model: {best_path.resolve()}')
print(f'Selection record: {(models_dir / "medical_model_selection.json").resolve()}')